# Nghiên cứu cải tiến TickNets bằng cơ chế Attention CBAM trên bộ dữ liệu PlantVillage

Notebook này tập trung thực hiện thực nghiệm so sánh mô hình **TickNetA-SE (Baseline)** và mô hình cải tiến **TickNetA-CBAM** trên bộ dữ liệu **PlantVillage** bệnh lý thực vật có độ phân giải cao (224x224).

### Mục tiêu thực nghiệm:
1. Huấn luyện dòng mạng di động tối ưu **TickNet-A** sử dụng cơ chế chú ý kênh **SE** gốc làm chuẩn so sánh.
2. Huấn luyện phiên bản cải tiến tích hợp **CBAM** (kết hợp cả Kênh - Channel và Không gian - Spatial) với các tinh chỉnh (Spatial Kernel 3x3 và reduction ratio = 8) để nâng cao khả năng định vị đốm bệnh.
3. So sánh hiệu năng chi tiết (Accuracy, Precision, Recall, F1-Score, Confusion Matrix) để chứng minh tính hiệu quả của CBAM trên ảnh kích thước lớn.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import time
import os
import gc

# Thiết lập seed để đảm bảo tính tái lặp (Reproducibility)
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Kiểm tra thiết bị chạy (GPU / CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {device}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

## 1. Cấu hình Siêu tham số

In [ ]:
CONFIG = {
    'batch_size': 64,         # Vừa vặn để tránh lỗi tràn bộ nhớ (OOM) trên Kaggle GPU
    'epochs': 30,             # 30 epochs là đủ để so sánh tốc độ hội tụ và độ chính xác
    'learning_rate': 0.01,    # Giảm nhẹ LR cho ảnh lớn
    'weight_decay': 1e-4,
    'reduction_ratio': 8,     # Giảm từ 16 xuống 8 cho CBAM để tăng dung lượng học kênh
    'seed': 42
}
print("Cấu hình thực nghiệm:")
for k, v in CONFIG.items():
    print(f"  - {k}: {v}")

## 2. Chuẩn bị Dữ liệu PlantVillage

In [ ]:
# Định nghĩa các phép biến đổi ảnh (Data Augmentation) cho tập huấn luyện và kiểm tra
plant_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

plant_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Lớp wrapper để áp dụng các transform khác nhau cho tập huấn luyện và kiểm tra sau khi split
class MapDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, map_fn):
        self.dataset = dataset
        self.map_fn = map_fn

    def __getitem__(self, index):
        x, y = self.dataset[index]
        return self.map_fn(x), y

    def __len__(self):
        return len(self.dataset)

# Hàm tự động quét tìm đường dẫn dữ liệu trên Kaggle hoặc máy cục bộ
def find_plant_village_path():
    possible_paths = [
        '/kaggle/input/plantvillage-dataset/color',
        '/kaggle/input/plantvillage-dataset/Color',
        '/kaggle/input/plantvillage/PlantVillage',
        '/kaggle/input/plantvillage-dataset/plantvillage_deeplearning_paper_dataset/color',
        '../input/plantvillage-dataset/color',
        './data/plantvillage',
    ]
    for p in possible_paths:
        if os.path.exists(p) and len(os.listdir(p)) > 0:
            return p
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'color' in dirs:
                return os.path.join(root, 'color')
            if 'PlantVillage' in dirs:
                return os.path.join(root, 'PlantVillage')
    return None

plant_dir = find_plant_village_path()
if plant_dir is None:
    print("KHÔNG tìm thấy dataset PlantVillage trên Kaggle/Local.")
    print("Tự động khởi tạo dữ liệu giả lập (Dummy Dataset) để tránh lỗi cú pháp khi test trên local...")
    class DummyPlantDataset(torch.utils.data.Dataset):
        def __init__(self, size=128):
            self.size = size
        def __len__(self):
            return self.size
        def __getitem__(self, idx):
            return torch.randn(3, 224, 224), np.random.randint(0, 10)
            
    plant_train_loader = DataLoader(DummyPlantDataset(128), batch_size=16, shuffle=True)
    plant_test_loader = DataLoader(DummyPlantDataset(32), batch_size=16, shuffle=False)
    plant_classes = [f"Benh_La_{i}" for i in range(10)]
    num_plant_classes = 10
else:
    print(f"Đã phát hiện thấy thư mục dữ liệu PlantVillage tại: {plant_dir}")
    full_dataset = torchvision.datasets.ImageFolder(root=plant_dir)
    plant_classes = full_dataset.classes
    num_plant_classes = len(plant_classes)
    
    # Phân chia 80% train, 20% validation/test
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_sub, test_sub = random_split(
        full_dataset, 
        [train_size, test_size], 
        generator=torch.Generator().manual_seed(CONFIG['seed'])
    )
    
    plant_train_dataset = MapDataset(train_sub, plant_train_transform)
    plant_test_dataset = MapDataset(test_sub, plant_test_transform)
    
    plant_train_loader = DataLoader(plant_train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    plant_test_loader = DataLoader(plant_test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    print(f"-> Tổng số lớp bệnh cây: {num_plant_classes}")
    print(f"-> Số mẫu huấn luyện: {len(plant_train_dataset)}")
    print(f"-> Số mẫu kiểm thử: {len(plant_test_dataset)}")

## 3. Định nghĩa Mô hình TickNets và các Module Chú ý (Attention)

In [ ]:
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class HSwish(nn.Module):
    def forward(self, x):
        return x * F.relu6(x + 3.0, inplace=True) / 6.0

class HSigmoid(nn.Module):
    def forward(self, x):
        return F.relu6(x + 3.0, inplace=True) / 6.0

def get_activation(activation):
    if activation == "relu":
        return nn.ReLU(inplace=True)
    elif activation == "relu6":
        return nn.ReLU6(inplace=True)
    elif activation == "swish":
        return Swish()
    elif activation == "hswish":
        return HSwish()
    elif activation == "sigmoid":
        return nn.Sigmoid()
    elif activation == "hsigmoid":
        return HSigmoid()
    else:
        raise NotImplementedError(f"Activation {activation} not implemented")

class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, dilation=1, groups=1, bias=False, use_bn=True, activation="relu"):
        super().__init__()
        self.use_bn = use_bn
        self.use_activation = (activation is not None)
        self.conv = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups, bias=bias)
        if self.use_bn:
            self.bn = nn.BatchNorm2d(num_features=out_channels)
        if self.use_activation:
            self.activation = get_activation(activation)
            
    def forward(self, x):
        x = self.conv(x)
        if self.use_bn:
            x = self.bn(x)
        if self.use_activation:
            x = self.activation(x)
        return x

def conv1x1_block(in_channels, out_channels, stride=1, groups=1, bias=False, use_bn=True, activation="relu"):
    return ConvBlock(in_channels=in_channels, out_channels=out_channels, kernel_size=1, stride=stride, padding=0, groups=groups, bias=bias, use_bn=use_bn, activation=activation)

def conv3x3_block(in_channels, out_channels, stride=1, bias=False, use_bn=True, activation="relu"):
    return ConvBlock(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=stride, padding=1, bias=bias, use_bn=use_bn, activation=activation)

def conv3x3_dw_blockAll(channels, stride=1, use_bn=True, activation="relu", padding=1, dilation=1):
    return ConvBlock(in_channels=channels, out_channels=channels, kernel_size=3, stride=stride, padding=padding, groups=channels, dilation=dilation, use_bn=use_bn, activation=activation)

class Classifier(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=in_channels, out_channels=num_classes, kernel_size=1, bias=True)
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return x
    def init_params(self):
        nn.init.xavier_normal_(self.conv.weight, gain=1.0)

# =========================================================================
# 1. Squeeze-and-Excitation (SE Attention)
# =========================================================================
class ChannelGate(nn.Module):
    def __init__(self, gate_channels, reduction_ratio=16):
        super(ChannelGate, self).__init__()        
        self.mlp = nn.Sequential(
            Flatten(),
            nn.Linear(gate_channels, gate_channels // reduction_ratio),
            nn.ReLU(),
            nn.Linear(gate_channels // reduction_ratio, gate_channels)
        )        
    def forward(self, x):
        squeeze_avg = F.avg_pool2d(x, (x.size(2), x.size(3)), stride=(x.size(2), x.size(3)))
        channel_att = self.mlp(squeeze_avg)
        scale = torch.sigmoid(channel_att).unsqueeze(2).unsqueeze(3).expand_as(x)
        return x * scale

class SE(nn.Module):
    def __init__(self, gate_channels, reduction_ratio=16):
        super(SE, self).__init__()
        self.ChannelGate = ChannelGate(gate_channels, reduction_ratio)
    def forward(self, x):
        return self.ChannelGate(x)

# =========================================================================
# 2. Convolutional Block Attention Module (CBAM) Cải tiến
# =========================================================================
class CBAMChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=8):
        super(CBAMChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
           
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class CBAMSpatialAttention(nn.Module):
    def __init__(self, kernel_size=3):  # Tinh chỉnh kernel 3x3 (thay vì 7x7) cho đốm lá cục bộ
        super(CBAMSpatialAttention, self).__init__()
        padding = 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_in = torch.cat([avg_out, max_out], dim=1)
        x_out = self.conv1(x_in)
        return self.sigmoid(x_out)

class CBAM_Tuned(nn.Module):
    def __init__(self, gate_channels, reduction_ratio=8):
        super(CBAM_Tuned, self).__init__()
        self.ChannelAttention = CBAMChannelAttention(gate_channels, reduction_ratio)
        self.SpatialAttention = CBAMSpatialAttention(kernel_size=3)

    def forward(self, x):
        x_out = x * self.ChannelAttention(x)
        x_out = x_out * self.SpatialAttention(x_out)
        return x_out

# =========================================================================
# 3. Khối tích chập FR-PDP kết hợp Cơ chế Attention
# =========================================================================
class FR_PDP_block(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.Pw1 = conv1x1_block(in_channels=in_channels, out_channels=in_channels, use_bn=False, activation=None)
        self.Dw = conv3x3_dw_blockAll(channels=in_channels, stride=stride)         
        self.Pw2 = conv1x1_block(in_channels=in_channels, out_channels=out_channels, groups=1)
        self.PwR = conv1x1_block(in_channels=in_channels, out_channels=out_channels, stride=stride)
        self.stride = stride
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.SE = SE(out_channels, 16)
        
    def forward(self, x):
        residual = x
        x = self.Pw1(x)        
        x = self.Dw(x)        
        x = self.Pw2(x)
        x = self.SE(x)
        if self.stride == 1 and self.in_channels == self.out_channels:
            x = x + residual
        else:            
            residual = self.PwR(residual)
            x = x + residual
        return x

class FR_PDP_CBAM_Tuned_block(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.Pw1 = conv1x1_block(in_channels=in_channels, out_channels=in_channels, use_bn=False, activation=None)
        self.Dw = conv3x3_dw_blockAll(channels=in_channels, stride=stride)         
        self.Pw2 = conv1x1_block(in_channels=in_channels, out_channels=out_channels, groups=1)
        self.PwR = conv1x1_block(in_channels=in_channels, out_channels=out_channels, stride=stride)
        self.stride = stride
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.CBAM = CBAM_Tuned(out_channels, CONFIG['reduction_ratio'])
        self.dropout = nn.Dropout2d(p=0.1)  # Thêm Dropout 10% chống overfitting
        
    def forward(self, x):
        residual = x
        x = self.Pw1(x)
        x = self.Dw(x)
        x = self.Pw2(x)
        x = self.CBAM(x)
        x = self.dropout(x)
        if self.stride == 1 and self.in_channels == self.out_channels:
            x = x + residual
        else:
            residual = self.PwR(residual)
            x = x + residual
        return x

# =========================================================================
# 4. Mạng TickNet-A Backbone
# =========================================================================
class TickNetA(nn.Module):
    def __init__(self, num_classes, block_class=FR_PDP_block, cifar=False):
        super().__init__()
        init_conv_channels = 32
        # Kênh đàn hồi theo hình dấu tick ✓
        channels = [[128], [64, 128], [256, 512, 128], [64, 128, 256], [512]]
        
        if cifar:
            self.in_size = (32, 32)
            init_conv_stride = 1
            strides = [1, 1, 2, 2, 2]
        else:
            self.in_size = (224, 224) # Cho kích thước PlantVillage
            init_conv_stride = 2
            strides = [2, 1, 2, 2, 2] # Thích nghi giảm kích thước xuống 7x7 trước pooling
            
        self.backbone = nn.Sequential()
        self.backbone.add_module("data_bn", nn.BatchNorm2d(num_features=3))
        self.backbone.add_module("init_conv", conv3x3_block(in_channels=3, out_channels=init_conv_channels, stride=init_conv_stride))
        
        in_ch = init_conv_channels
        for stage_id, stage_channels in enumerate(channels):
            stage = nn.Sequential()
            for unit_id, unit_channels in enumerate(stage_channels):
                stride = strides[stage_id] if unit_id == 0 else 1                
                stage.add_module(f"unit{unit_id + 1}", block_class(in_channels=in_ch, out_channels=unit_channels, stride=stride))
                in_ch = unit_channels
            self.backbone.add_module(f"stage{stage_id + 1}", stage)
            
        self.final_conv_channels = 1024        
        self.backbone.add_module("final_conv", conv1x1_block(in_channels=in_ch, out_channels=self.final_conv_channels, activation="relu"))
        self.backbone.add_module("global_pool", nn.AdaptiveAvgPool2d(output_size=1))
        
        self.classifier = Classifier(in_channels=self.final_conv_channels, num_classes=num_classes)
        self.init_params()

    def init_params(self):
        for name, module in self.backbone.named_modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
        self.classifier.init_params()

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 4. Quy trình Huấn luyện & Đánh giá

In [ ]:
def clear_memory(model=None):
    if model is not None:
        try:
            model.cpu()
        except:
            pass
        del model
    gc.collect()
    torch.cuda.empty_cache()

def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Mixed precision autocast (Huấn luyện chính xác hỗn hợp tăng tốc GPU)
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

def train_model(model, train_loader, test_loader, epochs, lr, wd, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()
    
    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': []
    }
    
    start_time = time.time()
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == epochs - 1:
            print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
            
    elapsed_time = time.time() - start_time
    print(f"=> Kết thúc huấn luyện trong: {elapsed_time/60:.2f} phút.")
    return history

def get_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.numpy())
    return np.array(all_preds), np.array(all_targets)

def evaluate_detailed(model, loader, classes, dataset_name, model_name, device):
    preds, targets = get_predictions(model, loader, device)
    print()
    print(f"================ BÁO CÁO PHÂN LOẠI CHI TIẾT: {model_name} trên {dataset_name} ================")
    print(classification_report(targets, preds, target_names=classes, digits=4))
    
    # Tính các chỉ số vĩ mô (Macro)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, preds, average='macro')
    acc = 100.0 * np.sum(preds == targets) / len(targets)
    
    # Vẽ Confusion Matrix
    cm = confusion_matrix(targets, preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues') # Ẩn annot vì 38 nhãn quá dày đặc
    plt.title(f'Confusion Matrix: {model_name} ({dataset_name})')
    plt.ylabel('Nhãn thực tế')
    plt.xlabel('Nhãn dự đoán')
    plt.tight_layout()
    plt.show()
    
    return acc, precision * 100, recall * 100, f1 * 100

## 5. Chạy Thực nghiệm

In [ ]:
print("="*60)
print("HUÂN LUYỆN TICKNET-A (SE - Baseline) TRÊN PLANTVILLAGE")
print("="*60)
model_plant_se = TickNetA(num_classes=num_plant_classes, block_class=FR_PDP_block, cifar=False).to(device)
print(f"Tổng số tham số: {count_parameters(model_plant_se):,}")

history_plant_se = train_model(
    model=model_plant_se,
    train_loader=plant_train_loader,
    test_loader=plant_test_loader,
    epochs=CONFIG['epochs'],
    lr=CONFIG['learning_rate'],
    wd=CONFIG['weight_decay'],
    device=device
)

acc_p_se, p_p_se, r_p_se, f1_p_se = evaluate_detailed(
    model=model_plant_se, 
    loader=plant_test_loader, 
    classes=plant_classes, 
    dataset_name="PlantVillage", 
    model_name="TickNetA-SE (Baseline)", 
    device=device
)
clear_memory(model_plant_se)

In [ ]:
print("="*60)
print("HUÂN LUYỆN TICKNET-A (CBAM - Cải tiến) TRÊN PLANTVILLAGE")
print("="*60)
model_plant_cbam = TickNetA(num_classes=num_plant_classes, block_class=FR_PDP_CBAM_Tuned_block, cifar=False).to(device)
print(f"Tổng số tham số: {count_parameters(model_plant_cbam):,}")

history_plant_cbam = train_model(
    model=model_plant_cbam,
    train_loader=plant_train_loader,
    test_loader=plant_test_loader,
    epochs=CONFIG['epochs'],
    lr=CONFIG['learning_rate'],
    wd=CONFIG['weight_decay'],
    device=device
)

acc_p_cbam, p_p_cbam, r_p_cbam, f1_p_cbam = evaluate_detailed(
    model=model_plant_cbam, 
    loader=plant_test_loader, 
    classes=plant_classes, 
    dataset_name="PlantVillage", 
    model_name="TickNetA-CBAM (Cải tiến)", 
    device=device
)
clear_memory(model_plant_cbam)

## 6. Trực quan hóa Đường cong học tập (Learning Curves)

In [ ]:
def plot_history(history_se, history_cbam, dataset_name):
    epochs = range(1, len(history_se['train_loss']) + 1)
    plt.figure(figsize=(16, 6))
    
    # 1. Loss Curves
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history_se['train_loss'], 'b--', label='SE Train Loss')
    plt.plot(epochs, history_se['test_loss'], 'b-', label='SE Val Loss')
    plt.plot(epochs, history_cbam['train_loss'], 'r--', label='CBAM Train Loss')
    plt.plot(epochs, history_cbam['test_loss'], 'r-', label='CBAM Val Loss')
    plt.title(f'Đường cong Loss trên {dataset_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # 2. Accuracy Curves
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history_se['train_acc'], 'b--', label='SE Train Acc')
    plt.plot(epochs, history_se['test_acc'], 'b-', label='SE Val Acc')
    plt.plot(epochs, history_cbam['train_acc'], 'r--', label='CBAM Train Acc')
    plt.plot(epochs, history_cbam['test_acc'], 'r-', label='CBAM Val Acc')
    plt.title(f'Đường cong Accuracy trên {dataset_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

plot_history(history_plant_se, history_plant_cbam, "PlantVillage")

## 7. Bảng So sánh Hiệu năng trên PlantVillage

In [ ]:
import pandas as pd

summary_data = {
    "Mô hình": ["TickNetA-SE (Baseline)", "TickNetA-CBAM (Cải tiến)"],
    "Accuracy (%)": [acc_p_se, acc_p_cbam],
    "Precision (%)": [p_p_se, p_p_cbam],
    "Recall (%)": [r_p_se, r_p_cbam],
    "F1-Score (%)": [f1_p_se, f1_p_cbam]
}

df_summary = pd.DataFrame(summary_data)
print("BẢNG SO SÁNH HIỆU NĂNG TỔNG HỢP TRÊN PLANTVILLAGE:")
display(df_summary)

# Trực quan hóa so sánh
plt.figure(figsize=(10, 5))
df_melted = pd.melt(df_summary, id_vars="Mô hình", var_name="Chỉ số", value_name="Giá trị (%)")
ax = sns.barplot(data=df_melted, x="Chỉ số", y="Giá trị (%)", hue="Mô hình", palette="Set2")
plt.title("So sánh trực quan các chỉ số hiệu năng trên PlantVillage")
plt.ylim(min(acc_p_se, acc_p_cbam) - 5, 100)
plt.grid(axis='y', linestyle='--', alpha=0.5)
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.3),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()